# Multi-gNB scenario: two RFsim gNBs sharing one Open5GS core

This example combines the working monolithic and CU/DU examples:

- `gnb`: monolithic OAI gNB with ten RFsim UEs.
- `cu` + `du`: a second OAI gNB with ten RFsim UEs.
- One shared Open5GS core, subscriber database and monitoring stack.

The two radio links use distinct simulated RUs, gNB identities and cell identities.
Every UE has its own IMSI and is assigned to exactly one radio endpoint.
The split gNB's N3 probe is on the CU; the monolithic probe is on `gnb`.

Run cells in order. Provisioning creates real resources. The default layout has
**eight VMs**: core/monitoring, gNB, CU, DU and four UE workers (five UEs each).
Adjust worker count and VM flavor below for your available capacity. Twenty active
RFsim UEs are a new scale test; this example has only been checked offline.

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../..").resolve()
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "orchestrator @ git+https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git@main"
%pip install --upgrade "git+https://github.com/yassir63/nador.git@main" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r "{REPOSITORY / 'ansible' / 'requirements.yaml'}"


Edit the project, infrastructure, VM flavor and UE worker count here.
The experiment name includes a timestamp so this example does not reuse another run's experiment.
Keep the SLICES CLI authenticated in this environment.

In [ ]:
from datetime import UTC, datetime
import asyncio
import json

PROJECT = "post5g-beta"
EXPERIMENT = "multi-gnb-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
INFRA = "fr-sophia2-bi-vm1"
VM_FLAVOR = "large"
UES_PER_GNB = 10
UE_WORKERS = 4  # At least one; increase if UE workers run out of CPU or memory.

UE_GROUPS = {
    "monolithic": [f"mono_ue{i:02d}" for i in range(1, UES_PER_GNB + 1)],
    "split": [f"split_ue{i:02d}" for i in range(1, UES_PER_GNB + 1)],
}
CLIENTS = UE_GROUPS["monolithic"] + UE_GROUPS["split"]
UE_NODES = [f"node{i}" for i in range(5, 5 + UE_WORKERS)]
print(f"Experiment: {EXPERIMENT}; {4 + UE_WORKERS} VMs; {len(CLIENTS)} UEs")


The metamodel describes the network. `gnb` has gNB ID 1; `cu`/`du` share gNB ID 2.
A UE connects to the monolithic `gnb` or the split `du`. Both gNBs connect to the same AMF.
All twenty subscribers are defined before the database is populated.

In [ ]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.model_validate(
    {
        "ran": {
            "components": {
                "gnb": {
                    "type": "gnb",
                    "config": {"gnbId": 1, "nrCellId": 1, "physicalCellId": 0, "ru": "mono_ru"},
                },
                "cu": {"type": "cu", "config": {"gnbId": 2, "nrCellId": 2}},
                "du": {
                    "type": "du",
                    "config": {
                        "gnbId": 2,
                        "gnbDuId": 2,
                        "nrCellId": 2,
                        "physicalCellId": 1,
                        "cu_cp": "cu",
                        "cu_up": "cu",
                        "ru": "split_ru",
                    },
                },
            },
        },
        "rus": {"mono_ru": {"type": "simulated"}, "split_ru": {"type": "simulated"}},
        "ues": {
            name: {
                "config": {
                    "supi": f"00101{100 + index:010d}",
                    "gnbs": ["gnb" if name in UE_GROUPS["monolithic"] else "du"],
                    "ru": "mono_ru" if name in UE_GROUPS["monolithic"] else "split_ru",
                }
            }
            for index, name in enumerate(CLIENTS)
        },
    }
)


The deployment model chooses OAI/Open5GS and places workloads explicitly.
`node1` runs the shared core, monitoring and iperf server; `node2` runs `gnb`,
`node3` runs `cu`, and `node4` runs `du`. UEs are distributed across the UE workers.

In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.model_validate(
    {
        "params": {
            "provisioning": {
                "project": PROJECT,
                "experiment": EXPERIMENT,
                "infra": INFRA,
                "flavor": VM_FLAVOR,
                "extra_path": "~/slices/bin",
                "create_experiment": True,
            },
            "k8s": {
                "master_node": "node1",
                "local_kubeconfig_path": str(Path(f".k8s-{EXPERIMENT}.conf").resolve()),
            },
        },
        "nodes": {
            "node1": {"roles": ["core", "database"]},
            **{name: {"roles": ["ran"]} for name in ("node2", "node3", "node4")},
            **{name: {"roles": ["ue"]} for name in UE_NODES},
        },
        "ran": {
            "implementation": "oai",
            "components": {
                "gnb": {"type": "gnb", "node_name": "node2"},
                "cu": {"type": "cu", "node_name": "node3"},
                "du": {"type": "du", "node_name": "node4"},
            },
        },
        "ues": {
            name: {"type": "containerized", "implementation": "oai", "node_name": UE_NODES[index % len(UE_NODES)]}
            for index, name in enumerate(CLIENTS)
        },
    }
)


Create the VMs, then deploy Kubernetes. This is the first cell that provisions infrastructure.

In [ ]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision_nodes(nodes=["node1"])
await prov.provision()
print(prov.get_ansible_inventory())


In [ ]:
k8s = K8SManagement(preceding=prov)
await k8s.deploy()


Deploy the shared core with the same SMF/UPF metrics image and configuration templates
as the passing RFsim examples. Register all twenty subscribers before starting UEs.

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(preceding=k8s)
for component in core.deployment_model.core.components.values():
    component.node_name = "node1"


In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


Start the monolithic gNB and CU first, then the DU once its CU is running.
The local gNB/CU charts include the same metrics-parser sidecars as the existing examples.

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(preceding=core)


In [ ]:
# Select local OAI charts that include the metrics-parser sidecar.
from importlib.resources import files

oai_charts = files("nador") / "assets" / "ran" / "oai"
for name, component in ran.deployment_model.ran.components.items():
    if component.implementation == "oai" and component.type in ("gnb", "cu", "cu_cp", "cu_up", "du"):
        component.custom_chart_path = str(oai_charts / f"oai-{component.type}")
        component.custom_rendered_files = [
            str(oai_charts / "values.yaml.j2"),
            str(oai_charts / "config.yaml.j2"),
        ]


In [ ]:
await ran.prepare(components=["gnb", "cu"])
await ran.install(components=["gnb", "cu"])
await ran.deploy(components=["du"])


Install monitoring **before UE attachment** so the AMF/SMF sniffers observe registration
and session establishment. Both `gnb` and `cu` are packet targets; DU traffic reaches
N3 through its CU. `eth0` is used with the default pod network.

The OAI calculator provides monolithic MAC KPIs. The CU parser is also scraped,
but does not provide DU MAC statistics.

In [ ]:
from nador import MonitoringManagement
from nador.models import MonitoringDeployment

QUERIES_FILE = Path("queries.json")
monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)
await monitoring.deploy()


Optional: run this cell to open Grafana from your laptop with the current inventory connection.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = [
    "ssh",
    "-N",
    "-T",
    "-S",
    "none",
    "-o",
    "ControlMaster=no",
    "-o",
    "ControlPersist=no",
    "-o",
    "BatchMode=yes",
    "-o",
    "ExitOnForwardFailure=yes",
    "-o",
    "ConnectTimeout=10",
]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += [
    "-p",
    str(ssh.port),
    "-L",
    f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
    f"{ssh.user}@{ssh.hostname or control}",
]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))


Start the twenty UEs sequentially to avoid starting all RFsim clients at once.
Their IMSIs are already registered. The core/RAN manager chain stays separate from
monitoring; monitoring observes the resulting pods and sessions.

In [ ]:
from orchestrator.deployment import UEManagement

ue = UEManagement(preceding=ran)
for client in CLIENTS:
    print(f"Deploying {client}")
    await ue.deploy(components=[client], quiet=True)


Read the assigned UE tunnel addresses and install iperf3 on the server VM.
Each UE must have a distinct address. This is also where you can inspect the two groups
before choosing traffic for your custom scenario.

In [ ]:
from datetime import UTC, datetime
from nador.artifacts import ExperimentRun
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = "node1"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5
UE_INTERFACE = "oaitun_ue1"

server_node = ue.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(ue.deployment_model.params.ansible.runner_path) / "iperf-server")


await run_ansible_playbook(
    playbook=[
        {
            "name": "Install iperf3 on the server VM",
            "hosts": SERVER_NODE,
            "gather_facts": False,
            "become": True,
            "tasks": [
                {
                    "name": "Install iperf3",
                    "ansible.builtin.apt": {
                        "name": "iperf3",
                        "state": "present",
                        "update_cache": True,
                        "cache_valid_time": 3600,
                    },
                }
            ],
        }
    ],
    inventory=server_inventory,
    runner_path=server_runner,
    quiet=True,
)
addresses = await ue.run_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE], components=CLIENTS, quiet=True, flatten=False
)

ue_ips = {}
for name in CLIENTS:
    ipv4 = [a["local"] for device in json.loads(addresses[name]) for a in device["addr_info"] if a["family"] == "inet"]
    if len(ipv4) != 1:
        raise RuntimeError(f"{name}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[name] = ipv4[0]

if len(set(ue_ips.values())) != len(CLIENTS):
    raise ValueError("Select UEs with different tunnel addresses")

if not QUERIES_FILE.is_file():
    raise FileNotFoundError(QUERIES_FILE)

print(f"iperf server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")


Traffic helpers use the same UE-to-VM TCP commands as the passing RFsim examples.
Uplink sends from the UE; downlink adds iperf's reverse mode.

In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60", "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await ue.run_command(
        command=[
            "iperf3",
            "-c",
            server_ip,
            "-B",
            ue_ips[client],
            "--bind-dev",
            UE_INTERFACE,
            "-p",
            str(port),
            "-t",
            str(duration),
            "--connect-timeout",
            "10000",
            "-J",
            "--get-server-output",
            *reverse,
        ],
        components=[client],
        quiet=True,
        flatten=False,
    )
    return json.loads(output[client])


Define the measured workload here. By default, every UE runs 20 seconds uplink and
20 seconds downlink, sequentially: **40 steps, 13 minutes 20 seconds of requested traffic**,
plus command and collection overhead. For a short smoke run, set
`TRAFFIC_CLIENTS = [UE_GROUPS["monolithic"][0], UE_GROUPS["split"][0]]`.
All twenty UEs remain deployed regardless of this traffic selection.

In [ ]:
TRAFFIC_CLIENTS = CLIENTS
SCENARIO = [
    {"name": f"{client}_{direction}", "client": client, "direction": direction, "duration": DURATION}
    for client in TRAFFIC_CLIENTS
    for direction in ("uplink", "downlink")
]


Run the workload and collect artifacts. `ExperimentRun` saves the timeline, query exports
and iperf reports. The group and address files identify which gNB each client used.

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="multi-gnb",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)
    experiment.save_json("ue_groups.json", UE_GROUPS)
    experiment.save_json("ue_addresses.json", ue_ips)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        await start_iperf_server(port)

        with experiment.section(test["name"]):
            report = await iperf_client(
                client=test["client"],
                direction=test["direction"],
                duration=test["duration"],
                port=port,
            )

        experiment.save_json(f"{test['name']}.json", report)
        if report.get("error"):
            raise RuntimeError(f"{test['name']}: {report['error']}")

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")


In [ ]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")


When finished, run the next cell to destroy this example's VMs and their workloads.
If you want to keep 5G running and remove only monitoring, run
`await monitoring.remove()` instead and leave the VM-cleanup cell unexecuted.

In [ ]:
await prov.unprovision()
